Intalação das Bibliotecas

In [34]:
!pip install -qU langchain langchain-community langchain-openai faiss-cpu



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
from pathlib import Path
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from openai import OpenAI

dotenv_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=dotenv_path, override=True)
 
api_key = os.getenv("OPENAI_API_KEY", "").strip()
 
if not api_key:
    raise RuntimeError("OPENAI_API_KEY não foi encontrado no arquivo .env.")
 
if not api_key.startswith("sk-"):
    raise RuntimeError("OPENAI_API_KEY não é válido. Certifique-se de que a chave começa com 'sk-'.")

print("Chave de API encontrada")

Chave de API encontrada


In [37]:
# download das fontes de informação
from langchain_community.document_loaders import PyPDFLoader

politica_1 = PyPDFLoader("Quantum_Commerce_Politica_de_Devolucao.pdf").load()
politica_2 = PyPDFLoader("Quantum_Commerce_Politica_de_Reembolso.pdf").load()

documento = politica_1 + politica_2


In [38]:
from langchain_openai import OpenAIEmbeddings

embeddings_model = OpenAIEmbeddings(openai_api_key=api_key)

embeddings_model.model

'text-embedding-ada-002'

In [39]:
# quebrar os dados em chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

pedacos = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=100
).split_documents(documento)

In [40]:
len(pedacos)

23

In [41]:
pedacos[:2]

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-09-22T08:53:07-03:00', 'author': 'Quantum Commerce', 'keywords': '', 'moddate': '2026-09-22T08:53:07-03:00', 'subject': 'Proposta versão 1.2. Cláusulas organizadas por assunto; numeração de origem preservada com prefixo D.', 'title': 'Quantum Commerce - Política de Devolução', 'trapped': '/False', 'source': 'Quantum_Commerce_Politica_de_Devolucao.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='Quantum Commerce\nPolítica de Devolução\nVersão 1.2 | Proposta | 22/09/2026\nCláusula D-01 - Quantum Commerce - Objetivo e canais\nA Quantum Commerce estabelece esta política para a elegibilidade, a solicitação e a logística\nde devoluções de compras de consumidores finais realizadas no site, aplicativo, lojas físicas e\ndemais canais oficiais, inclusive pedidos retirados em loja. A Quantum Commerce aplica as\ncondições próprias de serviços regulados e contrat

In [42]:
embeddings_model.embed_query(pedacos[0].page_content)

[0.0032872408628463745,
 -0.0008562115835957229,
 0.006122592370957136,
 -0.031774964183568954,
 -0.02262844890356064,
 0.027724945917725563,
 -0.008419414050877094,
 -0.016444697976112366,
 -0.00398885877802968,
 -0.010886118747293949,
 0.012340319342911243,
 0.004848467651754618,
 -0.0026331902481615543,
 -0.001797364791855216,
 -0.009058174677193165,
 0.006893862038850784,
 0.014881771989166737,
 -0.03324275463819504,
 -0.012924717739224434,
 0.005558579694479704,
 0.014922544360160828,
 0.026841552928090096,
 0.006815715692937374,
 0.003628706093877554,
 0.007427295669913292,
 0.007828219793736935,
 0.031258516013622284,
 -0.01970645599067211,
 0.019108466804027557,
 -0.014773047529160976,
 0.025156311690807343,
 0.012007348239421844,
 -0.02132374420762062,
 0.000168078244314529,
 -0.01345475297421217,
 -0.007624360267072916,
 0.01962491311132908,
 0.0024751988239586353,
 0.017762992531061172,
 -0.04115251824259758,
 0.014827409759163857,
 0.027738535776734352,
 0.00666621839627623

In [43]:
from langchain_community.vectorstores import InMemoryVectorStore

vectorstore = InMemoryVectorStore.from_documents(
    documents=pedacos, embedding=embeddings_model
)

In [44]:
from re import search
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [47]:
retriever.invoke("Desistência de compra")

[Document(id='ca37d66b-316e-4489-beaf-c682fb61ab17', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-09-22T08:53:07-03:00', 'author': 'Quantum Commerce', 'keywords': '', 'moddate': '2026-09-22T08:53:07-03:00', 'subject': 'Proposta versão 1.2. Cláusulas organizadas por assunto; numeração de origem preservada com prefixo D.', 'title': 'Quantum Commerce - Política de Devolução', 'trapped': '/False', 'source': 'Quantum_Commerce_Politica_de_Devolucao.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='original quando disponível, mas não a exigirá como condição absoluta para direitos legais.\nUso prolongado, dano causado pelo consumidor ou falta de componentes poderão limitar\napenas o benefício comercial, mediante justificativa individual.'),
 Document(id='f9ab1fd1-5861-4469-8e2e-4feab2eeb3f9', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-09-22T08:53:07

In [48]:
query = "Qual o valor receberei de reembolso por produto danificado?"

In [49]:
query_embed = embeddings_model.embed_query(query)

In [50]:
query_embed

[-0.0004904042580164969,
 0.008931896649301052,
 0.0032301293686032295,
 -0.014882135204970837,
 -0.03277208283543587,
 0.01819072850048542,
 -0.009278449229896069,
 -0.011560463346540928,
 -0.000389462715247646,
 0.007441067602485418,
 0.0035178333055227995,
 0.005453295540064573,
 -1.1998302397842053e-05,
 -0.019733868539333344,
 -0.003900348674505949,
 -0.0014605873730033636,
 0.015784479677677155,
 0.005852157715708017,
 0.011567002162337303,
 -0.004433254711329937,
 -0.007166441064327955,
 0.0056265718303620815,
 -0.018439199775457382,
 0.005924083758145571,
 0.011122369207441807,
 -0.017066068947315216,
 0.02237551286816597,
 -0.009232677519321442,
 0.03609375283122063,
 -0.01863536238670349,
 0.04056623950600624,
 0.005355214700102806,
 -0.029241172596812248,
 -0.005786770489066839,
 -0.016229111701250076,
 0.010756200179457664,
 0.01067119650542736,
 0.023081693798303604,
 0.023343242704868317,
 0.002821459202095866,
 0.01779840514063835,
 0.025017157196998596,
 0.0363814570009

In [51]:
similar_chunks = retriever.invoke(query)
similar_chunks

[Document(id='93ca04c9-974d-4d24-b44f-db39e67493f8', metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-09-22T08:53:07-03:00', 'author': 'Quantum Commerce', 'keywords': '', 'moddate': '2026-09-22T08:53:07-03:00', 'subject': 'Proposta versão 1.2. Cláusulas organizadas por assunto; numeração de origem preservada com prefixo R.', 'title': 'Quantum Commerce - Política de Reembolso', 'trapped': '/False', 'source': 'Quantum_Commerce_Politica_de_Reembolso.pdf', 'total_pages': 4, 'page': 3, 'page_label': '4'}, page_content='de devoluções ou ausência de embalagem original.\nCláusula R-22 - Quantum Commerce - Revisão de decisão e reclamações\nA Quantum Commerce informará o motivo de eventual recusa e disponibilizará revisão pelo\nmesmo protocolo. A Quantum Commerce responderá à contestação em até 5 dias corridos,\nou no prazo legal menor, sem suspender prazos obrigatórios de restituição. A Quantum\nCommerce manterá disponíveis os canais ex

Resgatar os documentos similares no Banco

In [52]:
import textwrap
similar_texts = [pedaco.page_content for pedaco in similar_chunks]

for i, texto in enumerate(similar_texts, start=1):
    print(f"\n--- Componente {i} ---")
    print(texto)




--- Componente 1 ---
de devoluções ou ausência de embalagem original.
Cláusula R-22 - Quantum Commerce - Revisão de decisão e reclamações
A Quantum Commerce informará o motivo de eventual recusa e disponibilizará revisão pelo
mesmo protocolo. A Quantum Commerce responderá à contestação em até 5 dias corridos,
ou no prazo legal menor, sem suspender prazos obrigatórios de restituição. A Quantum
Commerce manterá disponíveis os canais externos de defesa do consumidor e resolução de
conflitos aplicáveis a cada país.
Cláusula R-23 - Quantum Commerce - Dados e segurança do reembolso
A Quantum Commerce coletará somente dados necessários à identificação da compra e à
restituição, com acesso restrito e retenção conforme obrigações legais locais. A Quantum
Commerce nunca solicitará senha bancária ou código de autenticação para reembolsar.
Cláusula R-24 - Quantum Commerce - Publicação e implantação
A Quantum Commerce publicará a versão aprovada e sua data de vigência, preservando

--- Componente 

In [53]:
from langchain_core.prompts import ChatPromptTemplate

prompt_consulta_seguro = ChatPromptTemplate.from_messages(
    [
        ("system", "Responda usando exclusivamente o conteúdo fornecido. \n\nContexto: \n{contexto}"),
        ("human", "{query}")
    ]
)

for mensagem in prompt_consulta_seguro.messages:
    print(f"\n--- {mensagem.__class__.__name__} ---")

    for linha in mensagem.prompt.template.splitlines():
        print(linha)


--- SystemMessagePromptTemplate ---
Responda usando exclusivamente o conteúdo fornecido. 

Contexto: 
{contexto}

--- HumanMessagePromptTemplate ---
{query}


In [54]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

modelo = ChatOpenAI(
    model="gpt-4.1-nano",
    temperature=0.5,
    api_key=api_key
)

def no_rag(pergunta):
  return modelo.invoke(pergunta)

def rag(pergunta):
  cadeia = prompt_consulta_seguro | modelo | StrOutputParser()
  # retrieve
  trechos = retriever.invoke(query)
  # augment
  contexto = "\n\n".join(um_trecho.page_content for um_trecho in trechos)
  # generate
  return cadeia.invoke({ "query": pergunta, "contexto": contexto})

In [55]:
query

'Qual o valor receberei de reembolso por produto danificado?'

In [56]:
import textwrap
print(textwrap.fill(no_rag(query).content, width=80))

Olá! Para poder ajudá-lo(a) a determinar o valor de reembolso por um produto
danificado, preciso de algumas informações adicionais:  1. Qual é o valor
original do produto? 2. O produto está dentro do prazo de garantia? 3. Você
possui nota fiscal ou comprovante de compra? 4. O dano torna o produto
inutilizável ou apenas apresenta defeito? 5. A loja ou fabricante oferece algum
procedimento específico para reembolso ou troca?  Com esses detalhes, poderei
orientar melhor sobre os possíveis valores de reembolso ou os passos a seguir.


In [57]:
import textwrap
print(textwrap.fill(rag(query), width=80))


A Quantum Commerce reembolsará o valor efetivamente pago pelos itens devolvidos,
com tributos cobrados na venda e descontos corretamente considerados. No caso de
devolução parcial, a restituição do frete será proporcional aos itens
devolvidos, pelo rateio registrado no pedido ou valor superior legalmente
devido, sem multa ou taxa de reposição.
